 First calls

Run FSL BET from Python, look at the command it becomes, and pass its output on.

In [ ]:
import math
import shlex
from pathlib import Path

import nibabel as nib
import niwrap
from nilearn import plotting
from niwrap import fsl

T1W = "../data/ds000001/sub-01/anat/sub-01_T1w.nii.gz"

RUNNER = "auto"  # change to "dry" if Docker doesn't work on your machine
runner = niwrap.use_dry() if RUNNER == "dry" else niwrap.use_auto(data_dir="../work/01")


def show(img, roi=None, **kwargs):
    if not Path(img).exists():
        print(f"(dry run, nothing to plot) {img}")
    elif roi is None:
        plotting.plot_anat(img, **kwargs)
    else:
        plotting.plot_roi(roi, bg_img=img, **kwargs)

# The command behind the call

A `DryRunner` builds the command without running it. `runner=` applies to this call only.

In [ ]:
dry = niwrap.DryRunner()
fsl.bet(infile=T1W, fractional_intensity=0.4, binary_mask=True, runner=dry)
print(shlex.join(dry.last_cargs))

`fractional_intensity=0.4` became `-f 0.4`, `binary_mask=True` became `-m`. The output prefix `img_bet` is the default.

# Run BET for real

`use_auto()` in the setup cell picked Docker if you have it, so `bet` runs in FSL's container image. The log shows the command.

In [ ]:
bet = fsl.bet(infile=T1W, binary_mask=True)

The result holds the output paths.

In [ ]:
print(bet.outfile)
print(bet.binary_mask)

In [ ]:
show(bet.outfile, title="BET: brain")

Arguments are checked before anything runs:

In [ ]:
try:
    fsl.bet(infile=T1W, fractional_intensity=40)
except niwrap.StyxValidationError as e:
    print(e)

# Docs in your editor

- Tools: type `fsl.` and press <kbd>Tab</kbd> (Jupyter) or wait for completions (VS Code).
- Parameters: inside `fsl.bet(` press <kbd>Shift</kbd>+<kbd>Tab</kbd> (Jupyter) or hover (VS Code).
- Outputs: type `bet.` and press <kbd>Tab</kbd>.

Or:

In [ ]:
fsl.bet?

**Try it (2 min):** find FSL FAST. Which parameter names its outputs? Which output is the hard segmentation?

# Into Python

Outputs are ordinary files. Brain volume from BET's mask with nibabel:

In [ ]:
if RUNNER != "dry":
    mask = nib.load(bet.binary_mask)
    voxel_ml = math.prod(mask.header.get_zooms()) / 1000
    print(f"Brain volume: {mask.get_fdata().sum() * voxel_ml:.0f} ml")

# Into another tool

Pass `bet.outfile` to FAST (about two minutes). Set `out_basename`, or `seg.tissue_class_map` will be `None`.

In [ ]:
seg = fsl.fast(in_files=[bet.outfile], out_basename="t1w")
seg.tissue_class_map

In [ ]:
show(bet.outfile, roi=seg.tissue_class_map, title="FAST: CSF / GM / WM")